In [1]:
!pip install -q transformers datasets accelerate sentencepiece

In [3]:
import torch

from datasets import load_dataset

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [4]:
import zipfile
import os

zip_path = "/content/Dataset.zip"
extract_dir = "/content/extracted_dataset/"

# Create extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Load the dataset from the extracted files.
# Assuming the zip file contains text files.
# If your dataset consists of other formats (e.g., JSON, CSV),
# you'll need to change 'text' to 'json' or 'csv' accordingly,
# and adjust the `data_files` argument.
dataset = load_dataset(
    'text', # Specify the type of dataset to load (e.g., 'text', 'json', 'csv')
    data_files=[os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith('.txt')]
    # If the zip contains files with other extensions, adjust the filter, or
    # for a single file, provide its explicit path, e.g., data_files=os.path.join(extract_dir, 'my_data.txt')
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9469
    })
})


In [5]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

# Filter out examples where input_ids might be empty or only contain padding tokens after tokenization
# Assuming tokenized_dataset is a DatasetDict containing a 'train' split
# For GPT2, pad_token is usually eos_token
pad_token_id = tokenizer.pad_token_id
tokenized_dataset["train"] = tokenized_dataset["train"].filter(
    lambda example: any(id != pad_token_id for id in example["input_ids"])
)

# Add a check to ensure the training dataset is not empty after filtering
if len(tokenized_dataset["train"]) == 0:
    raise ValueError("The training dataset is empty after tokenization and filtering. Please check your input text data or tokenizer settings.")

Map:   0%|          | 0/9469 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9469 [00:00<?, ? examples/s]

In [7]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [9]:
training_args = TrainingArguments(

    output_dir="./gpt2-model",

    # overwrite_output_dir=True, # This argument is no longer supported

    num_train_epochs=2,

    per_device_train_batch_size=2,

    save_steps=500,

    save_total_limit=2,

    logging_steps=100,

    prediction_loss_only=True,

    fp16=torch.cuda.is_available()
)

In [10]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    data_collator=data_collator
)

In [11]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,4.211573
200,4.035907
300,3.881533
400,3.929074
500,3.894576
600,3.828128
700,3.825204
800,3.851202
900,3.670398
1000,3.633880


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4736, training_loss=3.3425678707457878, metrics={'train_runtime': 666.3381, 'train_samples_per_second': 14.212, 'train_steps_per_second': 7.108, 'total_flos': 84503986560000.0, 'train_loss': 3.3425678707457878, 'epoch': 2.0})

In [12]:
trainer.save_model("./trained_gpt2")

tokenizer.save_pretrained("./trained_gpt2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./trained_gpt2/tokenizer_config.json', './trained_gpt2/tokenizer.json')

In [13]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./trained_gpt2",
    tokenizer="./trained_gpt2"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:
prompt = "Once upon a time"

output = generator(
    prompt,
    max_length=100,
    num_return_sequences=1,
    temperature=0.8,
    top_k=50,
    top_p=0.95
)

print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'top_k', 'temperature', 'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Once upon a time he came to himself. To him he was a shepherd, whose name was Paulina, and he had visited her every day, and he was pleased to see that she lived a quiet, kind-hearted life. He had loved her dearly, and he had loved her well. He had lived on the banks of the Puck, and had loved her as much as he loved her, and he loved her as well as he loved anyone else in this village. He was sorry to see his wife killed, and he would have spared her. He would not have killed her. He would not have done it to a girl of his own birth, or to a widow who was his own worst enemy. He would have killed her if he thought she was dead, and he would not have done it to anyone else. He would have done it to the most poor and meanhearted people, and he would not have done it to his own wife. He would not have done it to a woman who knew his daughter well, and who loved her deeply. He would not have done it to anyone else.” He could not have done it to his own wife, or to anyone else.” He must ha